# Module 5: LangSmith Engine — Close the Loop Automatically

> *"Your proactive agent engineer."* — LangSmith

In Module 4 you built the improvement loop **by hand**. **Engine automates the whole thing.** It watches your deployed agent's traces and runs five stages on its own:

1. **Detect** — find a *recurring* failure across many traces
2. **Diagnose** — trace the root cause into your connected GitHub source
3. **Fix** — open a pull request (code or prompt)
4. **Guard** — deploy an online evaluator so the issue can't silently regress
5. **Reopen** — if it resurfaces after being resolved, reopen it

Engine runs on three primitives: **traces** (what the agent did), **context** (why it did it), and **evals** (whether a fix holds) — every stage above is built on one of them.

To give Engine something to find, we run the deployed `deep_agent` (Module 3) through an **assistant** called `engine-demo` that swaps in a subtly broken search tool. Engine scans in the background (~20 min first pass, then on a cron), so a presenter primes it ahead. The rest of the module follows what Engine works from — **traces**, **context**, **evaluators** — then recaps.

<img src="../images/adlc.avif" style="width: auto; max-height: 360px; border-radius: 8px;">

## Setup

Deployed runs trace to the deployment's own project — **`modular-workshops-deep-agent`** — so that's the project Engine analyzes and the one we point traces, datasets, and evaluators at. The cell below resolves the deployment URL and upserts the `engine-demo` assistant (the broken-search config lives in `utils/engine.py`).

In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(dotenv_path=project_root / ".env", override=True)

from langsmith import Client
from langgraph_sdk import get_sync_client
from utils.engine import deployment_url, upsert_engine_assistant

project_name = "modular-workshops-deep-agent"
client = Client()
sdk = get_sync_client(url=deployment_url(project_name))
assistant_id = upsert_engine_assistant(sdk)
print("engine-demo assistant:", assistant_id)


def view_link(url, label="View in LangSmith"):
    """Print a labeled LangSmith UI link."""
    print(f"\n\U0001f517 {label}: {url}")


project_url = client.read_project(project_name=project_name).url
_time = "timeModel=%7B%22duration%22%3A%221d%22%7D"
traces_url = f"{project_url}?{_time}&tab=0"   # tab 0 = Traces
issues_url = f"{project_url}?{_time}&tab=4"   # tab 4 = Engine issues
handle = client._get_settings().tenant_handle
context_url = f"{client._host_url}/hub/{handle}/engine-demo-context" if handle else f"{client._host_url}/hub"

view_link(traces_url, "View project in LangSmith")

### Seed a few traces

Ask `engine-demo` a handful of questions. Because it uses the broken search tool, every answer comes back vague and ungrounded. A couple of traces is enough. Engine requires 2 or more traces to create an issue.

In [ ]:
research_prompts = [
    "What is LangGraph used for?",
    "What is LangSmith for?",
    "What is the difference between LangChain and LangGraph?",
]

for q in research_prompts:
    sdk.runs.create(None, assistant_id, input={"messages": [{"role": "user", "content": q}]})
    print("seeded:", q)

print(f"\nSeeded {len(research_prompts)} traces into project: {project_name}")

view_link(traces_url, "View traces in LangSmith")

### The issue Engine filed

For each recurring failure, Engine opens an **issue** with a title, severity, and root-cause description. Browse them in the **Engine** tab, or list them from the CLI:

In [ ]:
!langsmith project issues list --project "{project_name}" --format pretty

view_link(issues_url, "View issues in LangSmith")

## Anatomy of an Engine issue

An **issue** is Engine's unit of work — one recurring failure, with everything you need to act on it in a single panel. Open any issue in the **Engine** tab and you get:

- **Severity** — a priority of `Low`, `Medium`, or `High`, so you triage the worst failures first. You can override it (with a reason that feeds back into Engine).
- **Category** — the tags Engine assigns (e.g. `hallucination`, `wrong_tool`, `failed_error_recovery`), matched to the concerns you chose during setup. Filter and sort the issue list by these.
- **Summary** — a plain-language **diagnosis** at the top describing the problem and its impact.
- **Frequency** — how many **contributing traces** support the diagnosis, plus how recently it was last seen — the signal that this is a *pattern*, not a one-off.
- **Linked traces** — the exact production runs behind the issue; click through to inspect any of them.
- **Proposed fix** — a concrete code or prompt change. Apply it with **Open PR** (opens a pull request in your connected repo), or **Copy Fix Context** to hand it to a coding assistant.
- **Suggested evaluator** — a ready-to-deploy check that scores future traces and **reopens the issue** if the failure returns.
- **Offline examples** — ground-truth dataset examples generated from the failing traces, for offline evaluation.

Each issue also carries a **status** — `open`, `resolved`, `ignored`, or `reopened` — that tracks it through the closed loop: detect → diagnose → fix → guard → auto-reopen on regression.

Those pieces map directly onto the three primitives the rest of this module walks through: the **linked traces** (1), the **proposed fix** rooted in your **context** (2), and the **suggested evaluator + offline examples** (3).

## 1. Traces

**Traces are the first primitive** — the raw record of what the agent actually did, and where Engine starts. It clusters recurring failures and links the exact runs behind each issue — open **Linked Traces** to inspect them. Here those runs show `easy_search` returning titles and URLs but **no page content**, so the agent answers without grounding. Peek at a few search outputs to see the gap:

In [ ]:
for r in client.list_runs(project_name=project_name, filter='eq(name, "easy_search")', limit=3):
    print("-", str(r.outputs)[:160])

view_link(traces_url, "View traces in LangSmith")

## 2. Context

**Context is the second primitive** — where a trace says *what* broke, context says *why*. Engine reads everything you connect — your **GitHub source** plus the agent's runtime context (prompts, skills, memory).

This assistant loads its guidance from the Context Hub repo `engine-demo-context`. Pull it straight from the repo the agent reads at runtime:

In [ ]:
from deepagents.backends.context_hub import ContextHubBackend

hub = ContextHubBackend("engine-demo-context")
print(hub.read("AGENTS.md").file_data["content"])

view_link(context_url, "View context in LangSmith")

That guidance is deliberately weak — *"lead with your own knowledge," "one search is enough," "don't cite sources"* — which, paired with the broken tool, pushes the agent toward confident, ungrounded answers.

Engine will include a suggested update to this context in the issue it files. 

## 3. Evaluators

**Evals are the third primitive** — they turn a one-time fix into a standing guarantee that the gap stays closed. Best practice splits evals by *when* they run: **offline** before you ship (against a curated dataset, to compare versions and catch regressions) and **online** after you ship (against live production traffic, to detect issues in real time). Engine sets up both from a resolved issue in one click.

### 3a. Offline evals — prove the fix before shipping

Engine's offline output is a **suggested evaluator** attached to the issue. To run it before shipping, point it at a dataset of real cases — best practice: *build datasets from real production traces, not synthetic examples.*

The cell below creates a small seed dataset we can run Engine's suggested evaluator against with `client.evaluate(...)` (the offline loop from Module 4) — a clean before/after on every prompt, model, or tool change.

In [ ]:
dataset_name = "engine-demo-evals"

if not client.has_dataset(dataset_name=dataset_name):
    ds = client.create_dataset(dataset_name)
    client.create_examples(dataset_id=ds.id, examples=[
        {"inputs": {"question": "What is LangGraph used for?"},
         "outputs": {"reference_answer": "Grounded explanation, citing sources, that LangGraph orchestrates stateful agent workflows."}},
        {"inputs": {"question": "What is LangSmith for?"},
         "outputs": {"reference_answer": "Grounded explanation, citing sources, of tracing, evals, and monitoring for LLM apps."}},
    ])
else:
    ds = client.read_dataset(dataset_name=dataset_name)

print("Dataset ready — run Engine's suggested evaluator against this with client.evaluate(...).")

view_link(ds.url, "View dataset in LangSmith")

### 3b. Online evals — guard live traffic against regression

The dataset proves the fix *today*. An **online evaluator** keeps it fixed *tomorrow*. Engine deploys a **run rule** that scores every new production trace as it lands — no reference outputs, just a reference-free LLM-as-judge / heuristic check — and **reopens the issue automatically if the regression returns**. This is the exact `create_run_rule(...)` workflow you wired by hand in Module 4 (filter to root traces, set a sampling rate to control cost), except Engine attaches it to the resolved issue in one click. That closes the loop: offline catches it before shipping, online catches it after.

**What a run rule looks like.** A rule is just a `filter` over incoming runs plus an action (attach a score, or reopen the issue). The filter uses the same query syntax as the traces view — `eq`, `and`, `gt`, and friends. The simplest checks are structural facts about the run, no LLM needed. For example, *did the run use the search tool?*

```python
# Match runs where the search tool was called.
filter = 'eq(name, "easy_search")'
```

Combine conditions to scope it — e.g. only slow search calls:

```python
filter = 'and(eq(name, "easy_search"), gt(latency, 3))'
```

When the check needs judgment rather than a structural fact — *"is this answer grounded in cited sources?"* — swap the heuristic for an LLM-as-judge prompt. That's the guard Engine deploys for our broken-search issue: scored on each root trace, reopening the issue if grounding regresses.

## Recap

Same loop you built by hand in Module 4 — now detected, diagnosed, fixed, and guarded automatically.

| Engine stage | By hand in Module 4 | Engine automates |
|---|---|---|
| **Detect** | `list_runs` + filters | Finds recurring failures on its own |
| **Diagnose** | Read traces, guessed | Traces the cause into GitHub source |
| **Fix** | Edited code yourself | Opens a PR (code or prompt) |
| **Guard** | `create_run_rule(...)` | Deploys an evaluator in one click |
| **Re-check** | Re-ran evals manually | Rescans every ~6h, auto-reopens regressions |


**Docs:** [LangSmith Engine](https://docs.langchain.com/langsmith/engine) · [Assistants](https://docs.langchain.com/langsmith/assistants) · [Engine webhooks](https://docs.langchain.com/langsmith/engine-webhooks)

In [ ]:
import json, subprocess

raw = subprocess.run(
    ["langsmith", "project", "issues", "list", "--project", project_name, "--format", "json"],
    capture_output=True, text=True,
).stdout
issues = json.loads(raw or "[]")

print(f"Engine filed {len(issues)} issue(s) for {project_name}:\n")
for i in issues:
    fix = f"PR #{i['fix_pr_number']}" if i.get("fix_pr_number") else f"draft branch {i['fix_branch']}"
    print(f"[sev {i['severity']}] {i['status']:6} · {i['name']}")
    print(f"   tags: {', '.join(i['tags']) or '—'} · fix: {fix}\n")

view_link(issues_url, "View issues in LangSmith")